In [2]:
from dj_notebook import activate

plus = activate()

Output()

In [3]:
from cuentas.models import User
from empresas.models import Empresa, UsuarioEmpresa, SucursalEmpresa
from django.db.models import Q
from django.contrib.contenttypes.models import ContentType

In [4]:
from ordentrabajo.models import (
    OrdenDeTrabajo, UsuarioAsignadoOT, DetalleTrabajo, 
    SeguimientoDetalleTrabajo, HistorialCambiosOrden, 
    AdjuntoDeOrden, DetalleGastoRendicionOT
)

def analizar_orden_trabajo(orden_id):
    """
    Extrae y muestra información detallada de una Orden de Trabajo
    
    Args:
        orden_id: ID de la orden de trabajo a analizar
    
    Returns:
        dict: Diccionario con toda la información de la OT
    """
    try:
        orden = OrdenDeTrabajo.objects.select_related(
            'empresa', 'cliente', 'responsable_empresa', 'solicitante_empresa',
            'responsable_empresa__usuario', 'solicitante_empresa__usuario'
        ).prefetch_related(
            'usuarioasignadoot_set__usuario_empresa__usuario',
            'detalletrabajo_set__tecnico_asignado__usuario',
            'historial__usuario__usuario',  # related_name="historial"
            'adjuntodeorden_set',
            'detallegastorendicionot_set__categoria'
        ).get(id=orden_id)
        
        # Información básica de la orden
        info = {
            'id': orden.id,
            'empresa': {
                'id': orden.empresa.id,
                'nombre': orden.empresa.nombre,
                'rut': orden.empresa.rut_empresa
            },
            'cliente': {
                'id': orden.cliente.id,
                'nombre': orden.cliente.nombre,
                'rut': orden.cliente.rut_empresa
            },
            'fechas': {
                'creacion': orden.fecha_creacion,
                'modificacion': orden.fecha_modificacion,
                'inicio_ot': orden.fecha_inicio_ot,
                'finalizacion_ot': orden.fecha_finalizacion_ot
            },
            'estado': orden.estado,
            'prioridad': orden.prioridad,
            'descripcion': orden.descripcion,
            'notas_internas': orden.notas_internas,
            'responsable': None,
            'solicitante': None,
            'usuarios_asignados': [],
            'detalles_trabajo': [],
            'historial_cambios': [],
            'adjuntos': [],
            'gastos_rendicion': []
        }
        
        # Responsable
        if orden.responsable_empresa:
            info['responsable'] = {
                'id': orden.responsable_empresa.id,
                'usuario_id': orden.responsable_empresa.usuario.id,
                'nombre': orden.responsable_empresa.usuario.get_nombre(),
                'email': orden.responsable_empresa.usuario.email
            }
        
        # Solicitante
        if orden.solicitante_empresa:
            info['solicitante'] = {
                'id': orden.solicitante_empresa.id,
                'usuario_id': orden.solicitante_empresa.usuario.id,
                'nombre': orden.solicitante_empresa.usuario.get_nombre(),
                'email': orden.solicitante_empresa.usuario.email
            }
        
        # Usuarios asignados
        for asignado in orden.usuarioasignadoot_set.all():
            usuario_info = {
                'id': asignado.id,
                'fecha_creacion': asignado.fecha_creacion
            }
            if asignado.usuario_empresa:
                usuario_info['tipo'] = 'interno'
                usuario_info['nombre'] = asignado.usuario_empresa.usuario.get_nombre()
                usuario_info['email'] = asignado.usuario_empresa.usuario.email
            else:
                usuario_info['tipo'] = 'externo'
                usuario_info['nombre'] = asignado.usuario_externo
                usuario_info['email'] = asignado.correo_usuario_externo
            info['usuarios_asignados'].append(usuario_info)
        
        # Detalles de trabajo
        for detalle in orden.detalletrabajo_set.all():
            detalle_info = {
                'id': detalle.id,
                'nombre': detalle.nombre,
                'descripcion': detalle.descripcion,
                'estado': detalle.estado,
                'fecha_creacion': detalle.fecha_creacion,
                'tecnico_asignado': None,
                'trabajo_relacionado': None,
                'insumo': None,
                'seguimientos': []
            }
            
            # Técnico asignado
            if detalle.tecnico_asignado:
                detalle_info['tecnico_asignado'] = {
                    'id': detalle.tecnico_asignado.id,
                    'nombre': detalle.tecnico_asignado.usuario.get_nombre(),
                    'email': detalle.tecnico_asignado.usuario.email
                }
            
            # Trabajo relacionado (GenericForeignKey)
            if detalle.trabajo:
                detalle_info['trabajo_relacionado'] = {
                    'tipo': detalle.content_type.model,
                    'id': detalle.trabajo_id
                }
            
            # Insumo (GuiaSalida)
            if detalle.insumo:
                detalle_info['insumo'] = {
                    'id': detalle.insumo.id,
                    'numero': getattr(detalle.insumo, 'numero', None)
                }
            
            # Seguimientos del detalle - cargamos sin prefetch para evitar conflictos
            seguimientos = SeguimientoDetalleTrabajo.objects.filter(
                detalle_trabajo=detalle
            ).select_related('usuario__usuario')
            
            for seguimiento in seguimientos:
                seguimiento_info = {
                    'id': seguimiento.id,
                    'tipo': seguimiento.tipo,
                    'fecha': seguimiento.fecha,
                    'comentario': seguimiento.comentario,
                    'usuario': None
                }
                if seguimiento.usuario:
                    seguimiento_info['usuario'] = {
                        'id': seguimiento.usuario.id,
                        'nombre': seguimiento.usuario.usuario.get_nombre()
                    }
                detalle_info['seguimientos'].append(seguimiento_info)
            
            info['detalles_trabajo'].append(detalle_info)
        
        # Historial de cambios - usar related_name="historial"
        for cambio in orden.historial.all():
            cambio_info = {
                'id': cambio.id,
                'fecha_cambio': cambio.fecha_cambio,
                'estado_anterior': cambio.estado_anterior,
                'estado_actual': cambio.estado_actual,
                'comentario': cambio.comentario,
                'usuario': {
                    'id': cambio.usuario.id,
                    'nombre': cambio.usuario.usuario.get_nombre()
                }
            }
            info['historial_cambios'].append(cambio_info)
        
        # Adjuntos
        for adjunto in orden.adjuntodeorden_set.all():
            adjunto_info = {
                'id': adjunto.id,
                'tipo': adjunto.tipo,
                'descripcion': adjunto.descripcion,
                'archivo': adjunto.archivo.name if adjunto.archivo else None,
                'fecha_creacion': adjunto.fecha_creacion
            }
            info['adjuntos'].append(adjunto_info)
        
        # Gastos de rendición
        for gasto in orden.detallegastorendicionot_set.all():
            gasto_info = {
                'id': gasto.id,
                'categoria': {
                    'id': gasto.categoria.id,
                    'nombre': gasto.categoria.nombre
                },
                'detalle': gasto.detalle,
                'cantidad': gasto.cantidad,
                'monto_unitario': gasto.monto_unitario,
                'monto_total': gasto.monto_total,
                'fecha_gasto': gasto.fecha_gasto
            }
            info['gastos_rendicion'].append(gasto_info)
        
        # Resumen de estadísticas
        info['estadisticas'] = {
            'total_usuarios_asignados': len(info['usuarios_asignados']),
            'total_detalles_trabajo': len(info['detalles_trabajo']),
            'total_seguimientos': sum(len(d['seguimientos']) for d in info['detalles_trabajo']),
            'total_cambios': len(info['historial_cambios']),
            'total_adjuntos': len(info['adjuntos']),
            'total_gastos': len(info['gastos_rendicion']),
            'suma_gastos': sum(g['monto_total'] for g in info['gastos_rendicion'])
        }
        
        return info
        
    except OrdenDeTrabajo.DoesNotExist:
        print(f"❌ No existe una Orden de Trabajo con ID {orden_id}")
        return None
    except Exception as e:
        print(f"❌ Error al analizar la orden: {str(e)}")
        import traceback
        traceback.print_exc()
        return None


def mostrar_resumen_ot(orden_id):
    """Muestra un resumen visual de la Orden de Trabajo"""
    info = analizar_orden_trabajo(orden_id)
    
    if not info:
        return
    
    print("=" * 80)
    print(f"📋 ORDEN DE TRABAJO #{info['id']}")
    print("=" * 80)
    print(f"\n🏢 EMPRESA: {info['empresa']['nombre']} (RUT: {info['empresa']['rut']})")
    print(f"👤 CLIENTE: {info['cliente']['nombre']} (RUT: {info['cliente']['rut']})")
    print(f"\n📊 ESTADO: {info['estado'].upper()}")
    print(f"⚡ PRIORIDAD: {info['prioridad']}")
    print(f"\n📝 DESCRIPCIÓN: {info['descripcion']}")
    
    if info['notas_internas']:
        print(f"\n🔒 NOTAS INTERNAS: {info['notas_internas']}")
    
    print(f"\n📅 FECHAS:")
    print(f"   • Creación: {info['fechas']['creacion']}")
    print(f"   • Inicio OT: {info['fechas']['inicio_ot'] or 'No definida'}")
    print(f"   • Finalización OT: {info['fechas']['finalizacion_ot'] or 'No definida'}")
    
    if info['responsable']:
        print(f"\n👔 RESPONSABLE: {info['responsable']['nombre']} ({info['responsable']['email']})")
    
    if info['solicitante']:
        print(f"🙋 SOLICITANTE: {info['solicitante']['nombre']} ({info['solicitante']['email']})")
    
    if info['usuarios_asignados']:
        print(f"\n👥 USUARIOS ASIGNADOS ({len(info['usuarios_asignados'])}):")
        for ua in info['usuarios_asignados']:
            tipo_icon = "🔧" if ua['tipo'] == 'interno' else "🌐"
            print(f"   {tipo_icon} {ua['nombre']} ({ua['email']})")
    
    if info['detalles_trabajo']:
        print(f"\n🔨 DETALLES DE TRABAJO ({len(info['detalles_trabajo'])}):")
        for dt in info['detalles_trabajo']:
            print(f"   • #{dt['id']}: {dt['nombre']} - Estado: {dt['estado']}")
            if dt['tecnico_asignado']:
                print(f"     Técnico: {dt['tecnico_asignado']['nombre']}")
            print(f"     Seguimientos: {len(dt['seguimientos'])}")
    
    if info['historial_cambios']:
        print(f"\n📜 HISTORIAL DE CAMBIOS ({len(info['historial_cambios'])}):")
        for cambio in info['historial_cambios'][:5]:  # Primeros 5
            print(f"   • {cambio['fecha_cambio']}: {cambio['usuario']['nombre']}")
            if cambio['estado_anterior']:
                print(f"     Antes: {cambio['estado_anterior'][:50]}...")
            if cambio['estado_actual']:
                print(f"     Después: {cambio['estado_actual'][:50]}...")
    
    if info['adjuntos']:
        print(f"\n📎 ADJUNTOS ({len(info['adjuntos'])}):")
        for adj in info['adjuntos']:
            print(f"   • {adj['tipo']}: {adj['descripcion'] or adj['archivo']}")
    
    if info['gastos_rendicion']:
        print(f"\n💰 GASTOS DE RENDICIÓN ({len(info['gastos_rendicion'])}):")
        for gasto in info['gastos_rendicion']:
            print(f"   • {gasto['categoria']['nombre']}: ${gasto['monto_total']:,}")
    
    print(f"\n📊 ESTADÍSTICAS:")
    print(f"   • Total Usuarios Asignados: {info['estadisticas']['total_usuarios_asignados']}")
    print(f"   • Total Detalles de Trabajo: {info['estadisticas']['total_detalles_trabajo']}")
    print(f"   • Total Seguimientos: {info['estadisticas']['total_seguimientos']}")
    print(f"   • Total Cambios Registrados: {info['estadisticas']['total_cambios']}")
    print(f"   • Total Adjuntos: {info['estadisticas']['total_adjuntos']}")
    print(f"   • Total Gastos: ${info['estadisticas']['suma_gastos']:,}")
    print("=" * 80)
    
    return info


# Ejemplo de uso:
# info = analizar_orden_trabajo(1)  # Devuelve diccionario completo
# mostrar_resumen_ot(1)  # Muestra resumen visual
print("✅ Funciones de análisis de OT cargadas:")
print("   • analizar_orden_trabajo(orden_id) - Extrae información completa en diccionario")
print("   • mostrar_resumen_ot(orden_id) - Muestra resumen visual en consola")

✅ Funciones de análisis de OT cargadas:
   • analizar_orden_trabajo(orden_id) - Extrae información completa en diccionario
   • mostrar_resumen_ot(orden_id) - Muestra resumen visual en consola


In [5]:
info = analizar_orden_trabajo(1)

In [6]:
# Mostrar resumen visual de la OT
mostrar_resumen_ot(1)

📋 ORDEN DE TRABAJO #1

🏢 EMPRESA: Snabbit (RUT: 11111111-1)
👤 CLIENTE: AYG ASOCIADOS (RUT: None)

📊 ESTADO: PENDIENTE
⚡ PRIORIDAD: 1

📝 DESCRIPCIÓN: Prueba OT1

📅 FECHAS:
   • Creación: 2025-11-07 19:56:45.377129+00:00
   • Inicio OT: 2025-11-07
   • Finalización OT: 2025-12-07

👔 RESPONSABLE: Juan Técnico (tecnico@snabbit.cl)
🙋 SOLICITANTE: Nathaly Aguilera (naguileran@aygasociados.cl)

👥 USUARIOS ASIGNADOS (1):
   🔧 Nathaly Aguilera (naguileran@aygasociados.cl)

🔨 DETALLES DE TRABAJO (1):
   • #1: Prueba Trabajo1 - Estado: pendiente
     Técnico: Juan Técnico
     Seguimientos: 1

📊 ESTADÍSTICAS:
   • Total Usuarios Asignados: 1
   • Total Detalles de Trabajo: 1
   • Total Seguimientos: 1
   • Total Cambios Registrados: 0
   • Total Adjuntos: 0
   • Total Gastos: $0


{'id': 1,
 'empresa': {'id': 1, 'nombre': 'Snabbit', 'rut': '11111111-1'},
 'cliente': {'id': 4, 'nombre': 'AYG ASOCIADOS', 'rut': None},
 'fechas': {'creacion': datetime.datetime(2025, 11, 7, 19, 56, 45, 377129, tzinfo=datetime.timezone.utc),
  'modificacion': datetime.datetime(2025, 11, 7, 19, 56, 45, 377129, tzinfo=datetime.timezone.utc),
  'inicio_ot': datetime.date(2025, 11, 7),
  'finalizacion_ot': datetime.date(2025, 12, 7)},
 'estado': 'pendiente',
 'prioridad': '1',
 'descripcion': 'Prueba OT1',
 'notas_internas': '',
 'responsable': {'id': 2,
  'usuario_id': 2,
  'nombre': 'Juan Técnico',
  'email': 'tecnico@snabbit.cl'},
 'solicitante': {'id': 5,
  'usuario_id': 5,
  'nombre': 'Nathaly Aguilera',
  'email': 'naguileran@aygasociados.cl'},
 'usuarios_asignados': [{'id': 1,
   'fecha_creacion': datetime.datetime(2025, 11, 7, 19, 56, 45, 405324, tzinfo=datetime.timezone.utc),
   'tipo': 'interno',
   'nombre': 'Nathaly Aguilera',
   'email': 'naguileran@aygasociados.cl'}],
 'det

In [7]:
# Probar con una de las órdenes recién creadas
mostrar_resumen_ot(5)  # OT en proceso


📋 ORDEN DE TRABAJO #5

🏢 EMPRESA: Snabbit (RUT: 11111111-1)
👤 CLIENTE: AYG ASOCIADOS (RUT: None)

📊 ESTADO: EN_PROCESO
⚡ PRIORIDAD: 2

📝 DESCRIPCIÓN: Orden de trabajo de prueba #1
Incluye mantenimiento preventivo y correctivo de equipos.
Se requiere revisión completa de sistemas.

🔒 NOTAS INTERNAS: Notas internas para OT #1. Cliente requiere atención especial.

📅 FECHAS:
   • Creación: 2025-11-12 22:56:49.110960+00:00
   • Inicio OT: 2025-11-02
   • Finalización OT: 2025-12-02

👔 RESPONSABLE: Fabian Huaiquiñir (fabian@gmail.com)
🙋 SOLICITANTE: Nathaly Aguilera (naguileran@aygasociados.cl)

👥 USUARIOS ASIGNADOS (3):
   🔧 Nathaly Aguilera (naguileran@aygasociados.cl)
   🔧 Juan Técnico (tecnico@snabbit.cl)
   🌐 Contratista Externo #1 (contratista1@external.com)

🔨 DETALLES DE TRABAJO (3):
   • #13: Actualización de Software #1 - Estado: pendiente
     Seguimientos: 3
   • #12: Reparación de Fallas #1 - Estado: en_proceso
     Técnico: Juan Técnico
     Seguimientos: 2
   • #11: Mantenimie

{'id': 5,
 'empresa': {'id': 1, 'nombre': 'Snabbit', 'rut': '11111111-1'},
 'cliente': {'id': 4, 'nombre': 'AYG ASOCIADOS', 'rut': None},
 'fechas': {'creacion': datetime.datetime(2025, 11, 12, 22, 56, 49, 110960, tzinfo=datetime.timezone.utc),
  'modificacion': datetime.datetime(2025, 11, 12, 22, 56, 49, 110960, tzinfo=datetime.timezone.utc),
  'inicio_ot': datetime.date(2025, 11, 2),
  'finalizacion_ot': datetime.date(2025, 12, 2)},
 'estado': 'en_proceso',
 'prioridad': '2',
 'descripcion': 'Orden de trabajo de prueba #1\nIncluye mantenimiento preventivo y correctivo de equipos.\nSe requiere revisión completa de sistemas.',
 'notas_internas': 'Notas internas para OT #1. Cliente requiere atención especial.',
 'responsable': {'id': 1,
  'usuario_id': 1,
  'nombre': 'Fabian Huaiquiñir',
  'email': 'fabian@gmail.com'},
 'solicitante': {'id': 5,
  'usuario_id': 5,
  'nombre': 'Nathaly Aguilera',
  'email': 'naguileran@aygasociados.cl'},
 'usuarios_asignados': [{'id': 11,
   'fecha_creaci

In [8]:
# Análisis de DetalleTrabajo para ver qué le falta
from ordentrabajo.models import DetalleTrabajo

detalle = DetalleTrabajo.objects.first()

print("=" * 80)
print("🔍 ANÁLISIS DE DETALLE DE TRABAJO")
print("=" * 80)
print(f"\n📋 Detalle ID: {detalle.id}")
print(f"Nombre: {detalle.nombre}")
print(f"Estado: {detalle.estado}")
print(f"Descripción: {detalle.descripcion}")

print(f"\n🔗 Relaciones:")
print(f"Orden: #{detalle.orden.id} - {detalle.orden.empresa.nombre}")
print(f"Técnico Asignado: {detalle.tecnico_asignado if detalle.tecnico_asignado else 'No asignado'}")

print(f"\n📦 Trabajo Relacionado (GenericForeignKey):")
print(f"ContentType: {detalle.content_type}")
print(f"Trabajo ID: {detalle.trabajo_id}")
print(f"Trabajo Objeto: {detalle.trabajo}")

print(f"\n📦 Insumo (GuiaSalida):")
print(f"Insumo: {detalle.insumo}")

print(f"\n💡 CAMPOS DISPONIBLES EN EL MODELO:")
print(f"   • nombre - ✅ Tiene")
print(f"   • orden - ✅ Tiene")
print(f"   • descripcion - ✅ Tiene")
print(f"   • content_type - ✅ Tiene (para trabajo relacionado)")
print(f"   • trabajo_id - ✅ Tiene (para trabajo relacionado)")
print(f"   • trabajo - ✅ Tiene (GenericForeignKey)")
print(f"   • estado - ✅ Tiene")
print(f"   • tecnico_asignado - ✅ Tiene")
print(f"   • insumo - ✅ Tiene (GuiaSalida)")

print(f"\n❌ CAMPOS QUE FALTAN O NO ESTÁN SIENDO USADOS:")
if not detalle.content_type or not detalle.trabajo_id:
    print(f"   1. content_type y trabajo_id están NULL")
    print(f"      → No hay trabajo relacionado (Cotización, VisitaSoporte, Compra)")
if not detalle.insumo:
    print(f"   2. insumo está NULL")
    print(f"      → No hay GuiaSalida vinculada")

print(f"\n🔧 TIPOS DE TRABAJO QUE PUEDE TENER:")
opciones_content_type = ContentType.objects.filter(
    Q(app_label='cotizaciones', model='cotizacion') |
    Q(app_label='visitas', model='visitasoporte') |
    Q(app_label='bodegas', model='compra')
)
for ct in opciones_content_type:
    print(f"   • {ct.app_label}.{ct.model}")

print(f"\n📊 Seguimientos del detalle:")
seguimientos = detalle.seguimientodetalletrabajo_set.all()
print(f"Total: {seguimientos.count()}")
for seg in seguimientos[:3]:
    print(f"   • {seg.fecha.strftime('%Y-%m-%d %H:%M')} - {seg.comentario[:50] if seg.comentario else 'Sin comentario'}...")

🔍 ANÁLISIS DE DETALLE DE TRABAJO

📋 Detalle ID: 30
Nombre: Seguimiento Visita Soporte
Estado: pendiente
Descripción: Trabajo derivado de visita previa

🔗 Relaciones:
Orden: #11 - Snabbit
Técnico Asignado: No asignado

📦 Trabajo Relacionado (GenericForeignKey):
ContentType: Visitas | Visita Soporte
Trabajo ID: 4
Trabajo Objeto: Visita a Empresa Cliente B de Snabbit

📦 Insumo (GuiaSalida):
Insumo: Guia de Salida 4 - Bodega: Bodega Principal

💡 CAMPOS DISPONIBLES EN EL MODELO:
   • nombre - ✅ Tiene
   • orden - ✅ Tiene
   • descripcion - ✅ Tiene
   • content_type - ✅ Tiene (para trabajo relacionado)
   • trabajo_id - ✅ Tiene (para trabajo relacionado)
   • trabajo - ✅ Tiene (GenericForeignKey)
   • estado - ✅ Tiene
   • tecnico_asignado - ✅ Tiene
   • insumo - ✅ Tiene (GuiaSalida)

❌ CAMPOS QUE FALTAN O NO ESTÁN SIENDO USADOS:

🔧 TIPOS DE TRABAJO QUE PUEDE TENER:
   • cotizaciones.cotizacion
   • visitas.visitasoporte
   • bodegas.compra

📊 Seguimientos del detalle:
Total: 2
   • 2025-11-

In [9]:
# Verificar los detalles de la OT #8 y #9 que tienen objetos relacionados
print("="*80)
print("🔍 VERIFICACIÓN DE OBJETOS RELACIONADOS EN DETALLES DE TRABAJO")
print("="*80)

# Analizar OT #8
orden8 = OrdenDeTrabajo.objects.get(id=8)
print(f"\n📋 ORDEN #{orden8.id} - {orden8.empresa.nombre} → {orden8.cliente.nombre}")
print(f"Estado: {orden8.estado}")

for detalle in orden8.detalletrabajo_set.all():
    print(f"\n  🔨 Detalle #{detalle.id}: {detalle.nombre}")
    print(f"     Estado: {detalle.estado}")
    
    # Trabajo relacionado
    if detalle.content_type and detalle.trabajo_id:
        print(f"     ✅ Trabajo Relacionado: {detalle.content_type.model} #{detalle.trabajo_id}")
        if detalle.trabajo:
            print(f"        Objeto: {detalle.trabajo}")
    else:
        print(f"     ❌ Sin trabajo relacionado")
    
    # Insumo
    if detalle.insumo:
        print(f"     ✅ Insumo (GuiaSalida): #{detalle.insumo.id}")
    else:
        print(f"     ❌ Sin insumo")
    
    # Técnico
    if detalle.tecnico_asignado:
        print(f"     👤 Técnico: {detalle.tecnico_asignado.usuario.get_nombre()}")

print("\n" + "="*80)

🔍 VERIFICACIÓN DE OBJETOS RELACIONADOS EN DETALLES DE TRABAJO

📋 ORDEN #8 - Snabbit → Empresa Cliente A
Estado: en_proceso

  🔨 Detalle #21: Seguimiento Visita Soporte
     Estado: pendiente
     ✅ Trabajo Relacionado: visitasoporte #1
        Objeto: Visita a Empresa Cliente A de Snabbit
     ✅ Insumo (GuiaSalida): #1

  🔨 Detalle #22: Instalación de Compra
     Estado: completado
     ✅ Trabajo Relacionado: compra #1
        Objeto: id: 1 - COMP-20251112200512-396
     ❌ Sin insumo
     👤 Técnico: Fabian Huaiquiñir

  🔨 Detalle #20: Instalación según Cotización
     Estado: en_proceso
     ✅ Trabajo Relacionado: cotizacion #2
        Objeto: Cotización #101 - Empresa Cliente A
     ❌ Sin insumo
     👤 Técnico: María Bodeguera



In [10]:
# Verificar estado de Retroalimentación en las OTs
from retroalimentacion.models import Retroalimentacion, RetroalimentacionAplicada
from core.models import PreguntaEnRetroalimentacion

print("="*80)
print("🔍 ANÁLISIS DE RETROALIMENTACIÓN")
print("="*80)

# Contar OTs con retroalimentación
total_ots = OrdenDeTrabajo.objects.count()
ots_con_retro = OrdenDeTrabajo.objects.filter(retroalimentacion__isnull=False).distinct().count()

print(f"\n📊 Estadísticas:")
print(f"   • Total OTs en BD: {total_ots}")
print(f"   • OTs con Retroalimentación: {ots_con_retro}")
print(f"   • OTs sin Retroalimentación: {total_ots - ots_con_retro}")

# Verificar preguntas configuradas
print(f"\n❓ Preguntas de Retroalimentación Configuradas:")
for ct_model in ['cotizacion', 'visitasoporte', 'compra']:
    try:
        ct = ContentType.objects.get(model=ct_model)
        preguntas = PreguntaEnRetroalimentacion.objects.filter(content_type=ct, activo=True)
        print(f"   • {ct.app_label}.{ct.model}: {preguntas.count()} preguntas")
        for p in preguntas[:3]:
            print(f"      - {p.texto[:60]}...")
    except ContentType.DoesNotExist:
        print(f"   • {ct_model}: ContentType no encontrado")

# Verificar retroalimentaciones existentes
print(f"\n📝 Retroalimentaciones Existentes:")
retros = Retroalimentacion.objects.all()[:5]
if retros:
    for retro in retros:
        print(f"   • OT #{retro.orden_trabajo.id}: {retro.uuid}")
        print(f"     Usuario: {retro.usuario_empresa or retro.usuario_externo or 'Sin usuario'}")
        print(f"     Fecha: {retro.fecha_retroalimentacion or 'Sin completar'}")
        print(f"     Preguntas aplicadas: {retro.retroalimentacion_aplicada.count()}")
else:
    print(f"   ❌ No hay retroalimentaciones creadas")

# Resumen de necesidades
print(f"\n💡 RESUMEN:")
if PreguntaEnRetroalimentacion.objects.filter(activo=True).count() == 0:
    print(f"   ⚠️  FALTAN: Preguntas de retroalimentación configuradas")
if ots_con_retro == 0:
    print(f"   ⚠️  FALTAN: Retroalimentaciones para las OTs")
    
if PreguntaEnRetroalimentacion.objects.filter(activo=True).count() > 0 and ots_con_retro > 0:
    print(f"   ✅ Sistema de retroalimentación completo")

print("="*80)

🔍 ANÁLISIS DE RETROALIMENTACIÓN

📊 Estadísticas:
   • Total OTs en BD: 11
   • OTs con Retroalimentación: 2
   • OTs sin Retroalimentación: 9

❓ Preguntas de Retroalimentación Configuradas:
   • cotizaciones.cotizacion: 3 preguntas
      - ¿Qué tan claro fue el detalle de la cotización?...
      - ¿El precio fue acorde a sus expectativas?...
      - ¿La respuesta fue oportuna?...
   • visitas.visitasoporte: 3 preguntas
      - ¿Cómo califica la atención del técnico?...
      - ¿Se resolvió el problema satisfactoriamente?...
      - ¿El tiempo de respuesta fue adecuado?...
   • bodegas.compra: 2 preguntas
      - ¿Los materiales llegaron en buen estado?...
      - ¿La calidad cumplió sus expectativas?...

📝 Retroalimentaciones Existentes:
   • OT #10: 15b46bba-801d-458c-92a6-734866e62ec8
     Usuario: Fabian Huaiquiñir
     Fecha: 2025-11-12 23:08:16.901376+00:00
     Preguntas aplicadas: 8
   • OT #11: 658166ae-6607-40dd-8c59-8a5e75dda9a2
     Usuario: Fabian Huaiquiñir
     Fecha: 202

In [11]:
# Análisis detallado de una retroalimentación completa
retro = Retroalimentacion.objects.get(orden_trabajo__id=10)

print("="*80)
print("📝 DETALLE DE RETROALIMENTACIÓN")
print("="*80)
print(f"\n🆔 UUID: {retro.uuid}")
print(f"📋 Orden: #{retro.orden_trabajo.id} - {retro.orden_trabajo.descripcion[:50]}...")
print(f"👤 Usuario: {retro.usuario_empresa.usuario.get_nombre()}")
print(f"📅 Fecha: {retro.fecha_retroalimentacion}")
print(f"💬 Observación: {retro.observacion_retroalimentacion}")

print(f"\n⭐ PREGUNTAS RESPONDIDAS ({retro.retroalimentacion_aplicada.count()}):")
for pregunta_aplicada in retro.retroalimentacion_aplicada.all():
    print(f"\n   📄 Modelo: {pregunta_aplicada.content_type.model} #{pregunta_aplicada.object_id}")
    print(f"   ❓ {pregunta_aplicada.pregunta.texto}")
    print(f"   ⭐ Calificación: {pregunta_aplicada.cantidad_estrellas}/5.0")
    if pregunta_aplicada.observaciones:
        print(f"   💭 {pregunta_aplicada.observaciones[:60]}...")

print("\n" + "="*80)

📝 DETALLE DE RETROALIMENTACIÓN

🆔 UUID: 15b46bba-801d-458c-92a6-734866e62ec8
📋 Orden: #10 - OT con objetos relacionados - 2025-11-12...
👤 Usuario: Fabian Huaiquiñir
📅 Fecha: 2025-11-12 23:08:16.901376+00:00
💬 Observación: Excelente servicio, todo según lo esperado

⭐ PREGUNTAS RESPONDIDAS (8):

   📄 Modelo: compra #3
   ❓ ¿Los materiales llegaron en buen estado?
   ⭐ Calificación: 4.2/5.0
   💭 Respuesta a: ¿Los materiales llegaron en bu......

   📄 Modelo: compra #3
   ❓ ¿La calidad cumplió sus expectativas?
   ⭐ Calificación: 4.1/5.0
   💭 Respuesta a: ¿La calidad cumplió sus expect......

   📄 Modelo: visitasoporte #3
   ❓ ¿Cómo califica la atención del técnico?
   ⭐ Calificación: 4.6/5.0
   💭 Respuesta a: ¿Cómo califica la atención del......

   📄 Modelo: visitasoporte #3
   ❓ ¿Se resolvió el problema satisfactoriamente?
   ⭐ Calificación: 4.3/5.0
   💭 Respuesta a: ¿Se resolvió el problema satis......

   📄 Modelo: visitasoporte #3
   ❓ ¿El tiempo de respuesta fue adecuado?
   ⭐ Cali

In [12]:
# Análisis de estructura de relaciones ManyToMany en OrdenDeTrabajo
print("="*80)
print("🔍 ANÁLISIS DE RELACIONES EN ORDENDETRABAJO")
print("="*80)

# Obtener los campos ManyToMany del modelo
from django.db.models import ManyToManyField

ot_fields = OrdenDeTrabajo._meta.get_fields()
m2m_fields = [f for f in ot_fields if isinstance(f, ManyToManyField)]

print(f"\n📊 Campos ManyToMany declarados en OrdenDeTrabajo:")
for field in m2m_fields:
    print(f"\n   • {field.name}:")
    print(f"     - Tipo: {field.__class__.__name__}")
    print(f"     - Through: {field.remote_field.through.__name__}")
    print(f"     - Related model: {field.remote_field.model.__name__}")
    
    # Verificar si la relación es self (problema potencial)
    if field.remote_field.model == OrdenDeTrabajo:
        print(f"     ⚠️  PROBLEMA: ManyToMany a 'self' detectado")
        print(f"     💡 Debería apuntar al modelo correcto, no a OrdenDeTrabajo")

print(f"\n" + "="*80)
print(f"❌ PROBLEMAS IDENTIFICADOS:")
print(f"="*80)

problemas = []

# Problema 1: usuarios_asignados apunta a 'self'
print(f"\n1️⃣  usuarios_asignados = models.ManyToManyField('self', ...)")
print(f"   ❌ INCORRECTO: Apunta a 'self' (OrdenDeTrabajo → OrdenDeTrabajo)")
print(f"   ✅ CORRECTO: Debería ser ManyToMany sin declaración explícita")
print(f"   💡 UsuarioAsignadoOT ya es ForeignKey a OrdenDeTrabajo")
problemas.append("usuarios_asignados apunta a 'self'")

# Problema 2: adjuntos apunta a 'self'
print(f"\n2️⃣  adjuntos = models.ManyToManyField('self', ...)")
print(f"   ❌ INCORRECTO: Apunta a 'self' (OrdenDeTrabajo → OrdenDeTrabajo)")
print(f"   ✅ CORRECTO: Debería ser ManyToMany sin declaración explícita")
print(f"   💡 AdjuntoDeOrden ya es ForeignKey a OrdenDeTrabajo")
problemas.append("adjuntos apunta a 'self'")

# Problema 3: trabajos apunta a 'self'
print(f"\n3️⃣  trabajos = models.ManyToManyField('self', ...)")
print(f"   ❌ INCORRECTO: Apunta a 'self' (OrdenDeTrabajo → OrdenDeTrabajo)")
print(f"   ✅ CORRECTO: Debería ser ManyToMany sin declaración explícita")
print(f"   💡 DetalleTrabajo ya es ForeignKey a OrdenDeTrabajo")
problemas.append("trabajos apunta a 'self'")

# Problema 4: historial_cambios apunta a 'self'
print(f"\n4️⃣  historial_cambios = models.ManyToManyField('self', ...)")
print(f"   ❌ INCORRECTO: Apunta a 'self' (OrdenDeTrabajo → OrdenDeTrabajo)")
print(f"   ✅ CORRECTO: Debería ser ManyToMany sin declaración explícita")
print(f"   💡 HistorialCambiosOrden ya es ForeignKey con related_name='historial'")
problemas.append("historial_cambios apunta a 'self'")

# Problema 5: seguimiento en DetalleTrabajo apunta a 'self'
print(f"\n5️⃣  DetalleTrabajo.seguimiento = models.ManyToManyField('self', ...)")
print(f"   ❌ INCORRECTO: Apunta a 'self' (DetalleTrabajo → DetalleTrabajo)")
print(f"   ✅ CORRECTO: Debería ser ManyToMany sin declaración explícita")
print(f"   💡 SeguimientoDetalleTrabajo ya es ForeignKey a DetalleTrabajo")
problemas.append("DetalleTrabajo.seguimiento apunta a 'self'")

print(f"\n" + "="*80)
print(f"📝 RESUMEN:")
print(f"="*80)
print(f"Total de problemas: {len(problemas)}")
print(f"\n💡 SOLUCIÓN:")
print(f"   Estas declaraciones ManyToMany con through='...' NO son necesarias")
print(f"   porque las tablas intermedias YA tienen ForeignKey a OrdenDeTrabajo.")
print(f"   Django automáticamente crea el acceso inverso con related_name.")
print(f"\n   Se pueden ELIMINAR todas estas líneas del modelo:")
for i, p in enumerate(problemas, 1):
    print(f"   {i}. {p}")

print("="*80)

🔍 ANÁLISIS DE RELACIONES EN ORDENDETRABAJO

📊 Campos ManyToMany declarados en OrdenDeTrabajo:

❌ PROBLEMAS IDENTIFICADOS:

1️⃣  usuarios_asignados = models.ManyToManyField('self', ...)
   ❌ INCORRECTO: Apunta a 'self' (OrdenDeTrabajo → OrdenDeTrabajo)
   ✅ CORRECTO: Debería ser ManyToMany sin declaración explícita
   💡 UsuarioAsignadoOT ya es ForeignKey a OrdenDeTrabajo

2️⃣  adjuntos = models.ManyToManyField('self', ...)
   ❌ INCORRECTO: Apunta a 'self' (OrdenDeTrabajo → OrdenDeTrabajo)
   ✅ CORRECTO: Debería ser ManyToMany sin declaración explícita
   💡 AdjuntoDeOrden ya es ForeignKey a OrdenDeTrabajo

3️⃣  trabajos = models.ManyToManyField('self', ...)
   ❌ INCORRECTO: Apunta a 'self' (OrdenDeTrabajo → OrdenDeTrabajo)
   ✅ CORRECTO: Debería ser ManyToMany sin declaración explícita
   💡 DetalleTrabajo ya es ForeignKey a OrdenDeTrabajo

4️⃣  historial_cambios = models.ManyToManyField('self', ...)
   ❌ INCORRECTO: Apunta a 'self' (OrdenDeTrabajo → OrdenDeTrabajo)
   ✅ CORRECTO: Debería

In [13]:
# Verificar que las relaciones siguen funcionando después de la corrección
print("="*80)
print("✅ VERIFICACIÓN POST-CORRECCIÓN")
print("="*80)

# Recargar el modelo
import importlib
import ordentrabajo.models
importlib.reload(ordentrabajo.models)
from ordentrabajo.models import OrdenDeTrabajo, DetalleTrabajo

# Probar con una OT existente
orden = OrdenDeTrabajo.objects.get(id=10)

print(f"\n📋 ORDEN #{orden.id} - {orden.cliente.nombre}")
print(f"Estado: {orden.estado}")

# Verificar acceso a usuarios asignados (reverse relation)
print(f"\n👥 Usuarios Asignados (usuarioasignadoot_set):")
usuarios = orden.usuarioasignadoot_set.all()
print(f"   Total: {usuarios.count()}")
for u in usuarios[:3]:
    nombre = u.usuario_empresa.usuario.get_nombre() if u.usuario_empresa else u.usuario_externo
    print(f"   • {nombre}")

# Verificar acceso a detalles (reverse relation)
print(f"\n🔨 Detalles de Trabajo (detalletrabajo_set):")
detalles = orden.detalletrabajo_set.all()
print(f"   Total: {detalles.count()}")
for d in detalles:
    print(f"   • #{d.id}: {d.nombre} - {d.estado}")

# Verificar acceso a historial (related_name='historial')
print(f"\n📜 Historial de Cambios (historial):")
historial = orden.historial.all()
print(f"   Total: {historial.count()}")
for h in historial[:2]:
    print(f"   • {h.fecha_cambio}: {h.usuario.usuario.get_nombre()}")

# Verificar acceso a adjuntos (reverse relation)
print(f"\n📎 Adjuntos (adjuntodeorden_set):")
adjuntos = orden.adjuntodeorden_set.all()
print(f"   Total: {adjuntos.count()}")
for a in adjuntos[:2]:
    print(f"   • {a.tipo}: {a.descripcion or 'Sin descripción'}")

# Verificar seguimientos en detalle (reverse relation)
detalle = detalles.first()
print(f"\n📊 Seguimientos del Detalle #{detalle.id} (seguimientodetalletrabajo_set):")
seguimientos = detalle.seguimientodetalletrabajo_set.all()
print(f"   Total: {seguimientos.count()}")
for s in seguimientos[:2]:
    print(f"   • {s.fecha.strftime('%Y-%m-%d %H:%M')}: {s.comentario[:40]}...")

print("\n" + "="*80)
print("✅ TODAS LAS RELACIONES FUNCIONAN CORRECTAMENTE")
print("="*80)

✅ VERIFICACIÓN POST-CORRECCIÓN

📋 ORDEN #10 - Empresa Cliente A
Estado: en_proceso

👥 Usuarios Asignados (usuarioasignadoot_set):
   Total: 4
   • Juan Técnico
   • Fabian Huaiquiñir
   • Juan Técnico

🔨 Detalles de Trabajo (detalletrabajo_set):
   Total: 3
   • #28: Instalación de Compra - completado
   • #27: Seguimiento Visita Soporte - pendiente
   • #26: Instalación según Cotización - en_proceso

📜 Historial de Cambios (historial):
   Total: 2
   • 2025-11-12 23:08:16.897861+00:00: Fabian Huaiquiñir
   • 2025-11-12 23:08:16.895861+00:00: Fabian Huaiquiñir

📎 Adjuntos (adjuntodeorden_set):
   Total: 3
   • documento: Adjunto de prueba #1
   • documento: Adjunto de prueba #2

📊 Seguimientos del Detalle #28 (seguimientodetalletrabajo_set):
   Total: 2
   • 2025-11-12 23:08: Seguimiento #1 para detalle 3...
   • 2025-11-12 23:08: Seguimiento #2 para detalle 3...

✅ TODAS LAS RELACIONES FUNCIONAN CORRECTAMENTE


c:\Users\LuisRojasMolina\.conda\envs\ENV-ERP\Lib\site-packages\django\db\models\base.py:366: RuntimeWarning: Model 'ordentrabajo.historicalordendetrabajo' was already registered. Reloading models is not advised as it can lead to inconsistencies, most notably with related models.
  new_class._meta.apps.register_model(new_class._meta.app_label, new_class)
c:\Users\LuisRojasMolina\.conda\envs\ENV-ERP\Lib\site-packages\django\db\models\base.py:366: RuntimeWarning: Model 'ordentrabajo.ordendetrabajo' was already registered. Reloading models is not advised as it can lead to inconsistencies, most notably with related models.
  new_class._meta.apps.register_model(new_class._meta.app_label, new_class)
c:\Users\LuisRojasMolina\.conda\envs\ENV-ERP\Lib\site-packages\django\db\models\base.py:366: RuntimeWarning: Model 'ordentrabajo.historicalusuarioasignadoot' was already registered. Reloading models is not advised as it can lead to inconsistencies, most notably with related models.
  new_class._me

In [23]:
# Validación de cierre de OT y creación de cierre administrativo
from ordentrabajo.utils import validar_cierre_ot, cerrar_ot
from empresas.models import UsuarioEmpresa
import traceback

print("="*80)
print("🔎 VALIDACIÓN DE CIERRE ADMINISTRATIVO")
print("="*80)

for ot_id in [10, 11]:
    try:
        res = validar_cierre_ot(ot_id)
        print(f"\n📋 OT #{ot_id} -> puede_cerrar={res['puede_cerrar']}")
        print(f"Validaciones: {res['validaciones']}")
        if res['observaciones']:
            print("Observaciones:")
            for obs in res['observaciones']:
                print(f"  - {obs}")
        # Intentar cierre cuando es posible
        if res['puede_cerrar']:
            cierre = cerrar_ot(ot_id, usuario_empresa=None, comentario="Cierre de prueba")
            print(f"✅ Cierre registrado: valido={cierre.valido} | id={cierre.id}")
        else:
            print("⚠️ No se registra cierre por validaciones pendientes.")
    except Exception as e:
        print(f"❌ Error procesando OT #{ot_id}: {e}")
        traceback.print_exc()

🔎 VALIDACIÓN DE CIERRE ADMINISTRATIVO

📋 OT #10 -> puede_cerrar=False
Validaciones: {'cotizacion': True, 'visitasoporte': True, 'compra': False}
Observaciones:
  - Compra no está en estado 'Completada'.; Compra sin ítems asociados.
⚠️ No se registra cierre por validaciones pendientes.

📋 OT #11 -> puede_cerrar=False
Validaciones: {'cotizacion': True, 'visitasoporte': True, 'compra': False}
Observaciones:
  - Compra no está en estado 'Completada'.; Compra sin ítems asociados.
⚠️ No se registra cierre por validaciones pendientes.


In [22]:
# Depuración: verificar _analizar_detalle_visita y traceback
from ordentrabajo.utils import _analizar_detalle_visita
from ordentrabajo.models import OrdenDeTrabajo
import traceback

try:
    o = OrdenDeTrabajo.objects.get(id=10)
    print(type(o), str(o))
    res = _analizar_detalle_visita(o)
    print("Resultado:", res)
except Exception as e:
    print("Se produjo un error en _analizar_detalle_visita:", e)
    traceback.print_exc()

<class 'ordentrabajo.models.OrdenDeTrabajo'> Orden #10 - Empresa Cliente A
Resultado: {'tipo': 'visitasoporte', 'tiene_retroalimentacion': True, 'preguntas_respondidas': 8, 'promedio_estrellas': 4.4, 'comentario': None}


In [17]:
# Sincronizar modelos tras reload: recargar retroalimentacion.models
import importlib
import retroalimentacion.models as retro_models
importlib.reload(retro_models)
print("retroalimentacion.models recargado")

retroalimentacion.models recargado


c:\Users\LuisRojasMolina\.conda\envs\ENV-ERP\Lib\site-packages\django\db\models\base.py:366: RuntimeWarning: Model 'retroalimentacion.retroalimentacion' was already registered. Reloading models is not advised as it can lead to inconsistencies, most notably with related models.
  new_class._meta.apps.register_model(new_class._meta.app_label, new_class)
c:\Users\LuisRojasMolina\.conda\envs\ENV-ERP\Lib\site-packages\django\db\models\base.py:366: RuntimeWarning: Model 'retroalimentacion.retroalimentacionaplicada' was already registered. Reloading models is not advised as it can lead to inconsistencies, most notably with related models.
  new_class._meta.apps.register_model(new_class._meta.app_label, new_class)
c:\Users\LuisRojasMolina\.conda\envs\ENV-ERP\Lib\site-packages\django\db\models\base.py:366: RuntimeWarning: Model 'retroalimentacion.logdeaccesoretroalimentacion' was already registered. Reloading models is not advised as it can lead to inconsistencies, most notably with related mod

In [21]:
# Recargar ordentrabajo.utils para aplicar últimos cambios
import importlib
import ordentrabajo.utils as ot_utils
importlib.reload(ot_utils)
print("ordentrabajo.utils recargado")

ordentrabajo.utils recargado


In [25]:
# Crear datos de ejemplo completos para probar cierres
from django.utils import timezone
from django.contrib.contenttypes.models import ContentType
from ordentrabajo.models import OrdenDeTrabajo, DetalleTrabajo
from empresas.models import Empresa, SucursalEmpresa, UsuarioEmpresa
from cuentas.models import User
from cotizaciones.models import Cotizacion
from visitas.models import VisitaSoporte
from bodegas.models import Bodega, Compra, ItemEnCompra, GuiaSalida
from items.models import ProveedorEmpresa, ItemEmpresa, Categoria
from core.models import PreguntaEnRetroalimentacion
from retroalimentacion.models import Retroalimentacion, RetroalimentacionAplicada
from ordentrabajo.utils import validar_cierre_ot, cerrar_ot


# Helpers

def get_or_create_basics():
    prestador, _ = Empresa.objects.get_or_create(
        nombre="Empresa Prestadora X",
        defaults={"direccion_principal":"Dir 1", "rut_empresa":"76.111.111-1"}
    )
    cliente_a, _ = Empresa.objects.get_or_create(
        nombre="Cliente A",
        defaults={"direccion_principal":"Dir A", "rut_empresa":"77.222.222-2"}
    )
    cliente_b, _ = Empresa.objects.get_or_create(
        nombre="Cliente B",
        defaults={"direccion_principal":"Dir B", "rut_empresa":"88.333.333-3"}
    )
    suc_prest, _ = SucursalEmpresa.objects.get_or_create(
        empresa=prestador,
        nombre="Casa Matriz",
        defaults={"direccion":"Central"}
    )
    # Usuario interno
    u, _ = User.objects.get_or_create(
        email="interno@prestadora.cl",
        defaults={"first_name":"Int", "last_name":"Erno", "is_active":True}
    )
    ue, _ = UsuarioEmpresa.objects.get_or_create(usuario=u, defaults={"sucursal":suc_prest})

    # Bodega y proveedor/ítem para compras
    bodega, _ = Bodega.objects.get_or_create(nombre="Bodega Central", sucursal=suc_prest)
    proveedor, _ = ProveedorEmpresa.objects.get_or_create(
        nombre="Proveedor ZX", rut="12.345.678-9", empresa=prestador
    )
    categoria, _ = Categoria.objects.get_or_create(nombre="Genérica")
    item, _ = ItemEmpresa.objects.get_or_create(
        nombre="Item Demo", empresa=prestador, defaults={"categoria":categoria}
    )

    return {
        "prestador": prestador,
        "cliente_a": cliente_a,
        "cliente_b": cliente_b,
        "suc_prest": suc_prest,
        "usuario_emp": ue,
        "bodega": bodega,
        "proveedor": proveedor,
        "item": item,
    }


def ensure_preguntas():
    modelos = [Cotizacion, VisitaSoporte, Compra]
    creadas = 0
    for m in modelos:
        ct = ContentType.objects.get_for_model(m)
        if not PreguntaEnRetroalimentacion.objects.filter(content_type=ct, activo=True).exists():
            PreguntaEnRetroalimentacion.objects.create(
                content_type=ct, texto=f"Califique la {ct.model} (1-5)"
            )
            PreguntaEnRetroalimentacion.objects.create(
                content_type=ct, texto=f"¿Comentarios sobre la {ct.model}?"
            )
            creadas += 2
    return creadas


def crear_detalle_cotizacion(ot, prestador, cliente):
    cot = Cotizacion.objects.create(
        nombre="Servicio Facturado",
        empresa=prestador,
        cliente=cliente,
        descripcion="Cotización facturada para cierre",
    )
    ct = ContentType.objects.get_for_model(Cotizacion)
    DetalleTrabajo.objects.create(
        orden=ot,
        nombre="Detalle Cotización",
        descripcion="Trabajo ligado a cotización facturada",
        content_type=ct,
        trabajo_id=cot.id,
        estado="completado",
    )
    return cot


def crear_detalle_visita(ot, prestador, cliente, usuario_emp):
    visita = VisitaSoporte.objects.create(
        empresa=prestador,
        cliente=cliente,
        descripcion_servicio="Visita realizada y conforme",
        estado="completada",
    )
    # Retroalimentación completa con preguntas respondidas
    retro = Retroalimentacion.objects.create(
        orden_trabajo=ot,
        usuario_empresa=usuario_emp,
        observacion_retroalimentacion="Todo OK",
        fecha_retroalimentacion=timezone.now(),
    )
    ct_v = ContentType.objects.get_for_model(VisitaSoporte)
    preguntas = PreguntaEnRetroalimentacion.objects.filter(content_type=ct_v, activo=True)
    if not preguntas:
        # fallback de seguridad
        preguntas = [PreguntaEnRetroalimentacion.objects.create(content_type=ct_v, texto="Satisfacción")]
    for p in preguntas:
        RetroalimentacionAplicada.objects.create(
            retroalimentacion=retro,
            content_type=ct_v,
            object_id=visita.id,
            pregunta=p,
            cantidad_estrellas=4.5,
            observaciones="Correcto"
        )
    ct = ContentType.objects.get_for_model(VisitaSoporte)
    DetalleTrabajo.objects.create(
        orden=ot,
        nombre="Detalle Visita",
        descripcion="Trabajo de visita con feedback",
        content_type=ct,
        trabajo_id=visita.id,
        estado="completado",
    )
    return visita


def crear_detalle_compra(ot, sucursal, usuario_emp, bodega, item):
    compra = Compra.objects.create(
        sucursal=sucursal,
        creado_por=usuario_emp,
        estado="1",  # Completada
        observaciones="Compra cerrada"
    )
    ItemEnCompra.objects.create(compra=compra, item=item, cantidad=2, precio=1000)
    guia = GuiaSalida.objects.create(bodega=bodega, creado_por=usuario_emp, estado="E")
    ct = ContentType.objects.get_for_model(Compra)
    DetalleTrabajo.objects.create(
        orden=ot,
        nombre="Detalle Compra",
        descripcion="Compra con ítems y guía entregada",
        content_type=ct,
        trabajo_id=compra.id,
        estado="completado",
        insumo=guia,
    )
    return compra, guia


# Orquestación principal
basics = get_or_create_basics()
creadas = ensure_preguntas()
print(f"Preguntas creadas: {creadas}")

creadas_ots = []

# 1) Uno de cada tipo
ot_cot = OrdenDeTrabajo.objects.create(
    empresa=basics["prestador"], cliente=basics["cliente_a"],
    descripcion="OT solo cotización", estado="completada"
)
crear_detalle_cotizacion(ot_cot, basics["prestador"], basics["cliente_a"])
creadas_ots.append(ot_cot.id)

ot_vis = OrdenDeTrabajo.objects.create(
    empresa=basics["prestador"], cliente=basics["cliente_a"],
    descripcion="OT solo visita", estado="completada"
)
crear_detalle_visita(ot_vis, basics["prestador"], basics["cliente_a"], basics["usuario_emp"])
creadas_ots.append(ot_vis.id)

ot_cmp = OrdenDeTrabajo.objects.create(
    empresa=basics["prestador"], cliente=basics["cliente_b"],
    descripcion="OT solo compra", estado="completada"
)
crear_detalle_compra(ot_cmp, basics["suc_prest"], basics["usuario_emp"], basics["bodega"], basics["item"])
creadas_ots.append(ot_cmp.id)

# 2) Un par de OTs con varios tipos (todas cerrables)
ot_multi1 = OrdenDeTrabajo.objects.create(
    empresa=basics["prestador"], cliente=basics["cliente_a"],
    descripcion="OT mixta 1", estado="completada"
)
crear_detalle_cotizacion(ot_multi1, basics["prestador"], basics["cliente_a"]) 
crear_detalle_visita(ot_multi1, basics["prestador"], basics["cliente_a"], basics["usuario_emp"])
crear_detalle_compra(ot_multi1, basics["suc_prest"], basics["usuario_emp"], basics["bodega"], basics["item"])
creadas_ots.append(ot_multi1.id)

ot_multi2 = OrdenDeTrabajo.objects.create(
    empresa=basics["prestador"], cliente=basics["cliente_b"],
    descripcion="OT mixta 2", estado="completada"
)
crear_detalle_compra(ot_multi2, basics["suc_prest"], basics["usuario_emp"], basics["bodega"], basics["item"])
crear_detalle_cotizacion(ot_multi2, basics["prestador"], basics["cliente_b"])  # cot facturada
crear_detalle_visita(ot_multi2, basics["prestador"], basics["cliente_b"], basics["usuario_emp"])
creadas_ots.append(ot_multi2.id)

print("OTs creadas:", creadas_ots)

# Validar y, si procede, cerrar
for oid in creadas_ots:
    res = validar_cierre_ot(oid)
    print(f"OT #{oid}: puede_cerrar={res['puede_cerrar']} -> {res['validaciones']}")
    if res["puede_cerrar"]:
        cierre = cerrar_ot(oid, comentario="Cierre automático de prueba")
        print(f"  Cierre creado: valido={cierre.valido} id={cierre.id}")
    else:
        print("  Observaciones:")
        for o in res["observaciones"]:
            print("   -", o)

Preguntas creadas: 0
OTs creadas: [12, 13, 14, 15, 16]
OT #12: puede_cerrar=True -> {'cotizacion': True, 'visitasoporte': True, 'compra': True}
  Cierre creado: valido=True id=1
OT #13: puede_cerrar=True -> {'cotizacion': True, 'visitasoporte': True, 'compra': True}
  Cierre creado: valido=True id=2
OT #14: puede_cerrar=True -> {'cotizacion': True, 'visitasoporte': True, 'compra': True}
  Cierre creado: valido=True id=3
OT #15: puede_cerrar=True -> {'cotizacion': True, 'visitasoporte': True, 'compra': True}
  Cierre creado: valido=True id=4
OT #16: puede_cerrar=True -> {'cotizacion': True, 'visitasoporte': True, 'compra': True}
  Cierre creado: valido=True id=5


In [27]:
# Auditoría de datos y creación de OTs coherentes para probar cierre administrativo
from django.utils import timezone
from django.contrib.contenttypes.models import ContentType
from ordentrabajo.models import OrdenDeTrabajo, DetalleTrabajo
from empresas.models import Empresa, SucursalEmpresa, UsuarioEmpresa
from cuentas.models import User
from cotizaciones.models import Cotizacion
from visitas.models import VisitaSoporte
from bodegas.models import Bodega, Compra, ItemEnCompra, GuiaSalida
from items.models import ProveedorEmpresa, ItemEmpresa, Categoria
from core.models import PreguntaEnRetroalimentacion
from retroalimentacion.models import Retroalimentacion, RetroalimentacionAplicada
from ordentrabajo.utils import validar_cierre_ot, cerrar_ot

print("="*80)
print("🔎 AUDITORÍA DE DATOS INICIALES")
print("="*80)

# 1) Empresas, sucursales, usuario, bodega, proveedor, item
prestador = Empresa.objects.order_by('id').first()
if not prestador:
    prestador = Empresa.objects.create(nombre="Empresa Prestadora X", direccion_principal="Dir 1", rut_empresa="76.111.111-1")
    print("Creada empresa prestadora:", prestador)
clientes = Empresa.objects.exclude(id=prestador.id)
if not clientes.exists():
    cliente = Empresa.objects.create(nombre="Cliente A", direccion_principal="Dir A", rut_empresa="77.222.222-2")
    print("Creado cliente:", cliente)
else:
    cliente = clientes.first()

suc = prestador.sucursales.first() or SucursalEmpresa.objects.create(empresa=prestador, nombre="Casa Matriz", direccion="Central")
print("Sucursal prestador:", suc.id)

usuario_emp = suc.usuarios.first()
if not usuario_emp:
    u = User.objects.filter(is_active=True).first() or User.objects.create(email="interno@prestadora.cl", first_name="Int", last_name="Erno", is_active=True)
    usuario_emp = UsuarioEmpresa.objects.create(usuario=u, sucursal=suc)
    print("Creado UsuarioEmpresa:", usuario_emp.id)

bodega = Bodega.objects.filter(sucursal=suc).first() or Bodega.objects.create(nombre="Bodega Central", sucursal=suc)
print("Bodega:", bodega.id)

proveedor = ProveedorEmpresa.objects.filter(empresa=prestador).first() or ProveedorEmpresa.objects.create(nombre="Proveedor ZX", rut="12.345.678-9", empresa=prestador)
cat = Categoria.objects.order_by('id').first() or Categoria.objects.create(nombre="Genérica")
item = ItemEmpresa.objects.filter(empresa=prestador).first() or ItemEmpresa.objects.create(nombre="Item Demo", empresa=prestador, categoria=cat)
print("Proveedor:", proveedor.id, "| Item:", item.id)

# 2) Preguntas de retro por modelo
creadas_preg = 0
for m in (Cotizacion, VisitaSoporte, Compra):
    ct = ContentType.objects.get_for_model(m)
    if not PreguntaEnRetroalimentacion.objects.filter(content_type=ct, activo=True).exists():
        PreguntaEnRetroalimentacion.objects.create(content_type=ct, texto=f"Califique la {ct.model} (1-5)")
        PreguntaEnRetroalimentacion.objects.create(content_type=ct, texto=f"¿Comentarios sobre la {ct.model}?")
        creadas_preg += 2
print("Preguntas creadas (si faltaban):", creadas_preg)

print("\n="*40)
print("🔧 CREACIÓN/REUTILIZACIÓN DE ENTIDADES ASOCIADAS")
print("="*80)

# 3) Crear o reutilizar objetos base por tipo
# Cotización (facturada)
cot = Cotizacion.objects.filter(empresa=prestador, cliente=cliente).exclude(fecha_facturacion__isnull=True).first()
if not cot:
    cot = Cotizacion.objects.create(nombre="Servicio Facturado", empresa=prestador, cliente=cliente, descripcion="Cotización facturada para cierre")
print("Cotización:", cot.id)

# Compra completa con ítems
compra = Compra.objects.filter(sucursal=suc, estado="1").first()
if not compra:
    compra = Compra.objects.create(sucursal=suc, creado_por=usuario_emp, estado="1", observaciones="Compra cerrada")
if not ItemEnCompra.objects.filter(compra=compra).exists():
    ItemEnCompra.objects.create(compra=compra, item=item, cantidad=2, precio=1000)
print("Compra:", compra.id, "| Ítems:", ItemEnCompra.objects.filter(compra=compra).count())

# Helper: obtener una guía de salida disponible (no usada por otro detalle) o crear una nueva
from django.db.models import Q

def next_insumo_disponible():
    usadas = DetalleTrabajo.objects.exclude(insumo__isnull=True).values_list("insumo_id", flat=True)
    guia = GuiaSalida.objects.filter(bodega=bodega, estado__in=("E","T")).exclude(id__in=list(usadas)).first()
    if not guia:
        guia = GuiaSalida.objects.create(bodega=bodega, creado_por=usuario_emp, estado="E")
    return guia

print("Guías disponibles antes:", GuiaSalida.objects.filter(bodega=bodega, estado__in=("E","T")).count())

# Visita
visita = VisitaSoporte.objects.filter(empresa=prestador, cliente=cliente).first()
if not visita:
    visita = VisitaSoporte.objects.create(empresa=prestador, cliente=cliente, descripcion_servicio="Visita correcta", estado="completada")
print("Visita:", visita.id)

print("\n="*40)
print("🧱 CREACIÓN DE OTs DE EJEMPLO")
print("="*80)

creadas_ots = []

# (A) OT solo cotización
ot_cot = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT solo cotización", estado="completada")
ct_cot = ContentType.objects.get_for_model(Cotizacion)
DetalleTrabajo.objects.create(orden=ot_cot, nombre="Detalle Cotización", descripcion="Ligado a cotización facturada", content_type=ct_cot, trabajo_id=cot.id, estado="completado")
creadas_ots.append(ot_cot.id)

# (B) OT solo visita (con retro)
ot_vis = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT solo visita", estado="completada")
ct_vis = ContentType.objects.get_for_model(VisitaSoporte)
DetalleTrabajo.objects.create(orden=ot_vis, nombre="Detalle Visita", descripcion="Visita con feedback", content_type=ct_vis, trabajo_id=visita.id, estado="completado")
# Crear retroalimentación para esta OT y aplicar preguntas
ct_v = ContentType.objects.get_for_model(VisitaSoporte)
pregs_v = list(PreguntaEnRetroalimentacion.objects.filter(content_type=ct_v, activo=True))
retro_vis = Retroalimentacion.objects.create(orden_trabajo=ot_vis, usuario_empresa=usuario_emp, observacion_retroalimentacion="OK", fecha_retroalimentacion=timezone.now())
for p in pregs_v:
    RetroalimentacionAplicada.objects.create(retroalimentacion=retro_vis, content_type=ct_v, object_id=visita.id, pregunta=p, cantidad_estrellas=4.5)
creadas_ots.append(ot_vis.id)

# (C) OT solo compra (completa + items + guía válida)
ot_cmp = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT solo compra", estado="completada")
ct_cmp = ContentType.objects.get_for_model(Compra)
DetalleTrabajo.objects.create(orden=ot_cmp, nombre="Detalle Compra", descripcion="Compra lista", content_type=ct_cmp, trabajo_id=compra.id, estado="completado", insumo=next_insumo_disponible())
creadas_ots.append(ot_cmp.id)

# (D) Mixtas: cot + visita + compra
ot_multi1 = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT mixta 1", estado="completada")
DetalleTrabajo.objects.create(orden=ot_multi1, nombre="Cot", descripcion="", content_type=ct_cot, trabajo_id=cot.id, estado="completado")
DetalleTrabajo.objects.create(orden=ot_multi1, nombre="Vis", descripcion="", content_type=ct_vis, trabajo_id=visita.id, estado="completado")
DetalleTrabajo.objects.create(orden=ot_multi1, nombre="Cmp", descripcion="", content_type=ct_cmp, trabajo_id=compra.id, estado="completado", insumo=next_insumo_disponible())
# Retro para la visita de esta OT
retro_m1 = Retroalimentacion.objects.create(orden_trabajo=ot_multi1, usuario_empresa=usuario_emp, observacion_retroalimentacion="OK", fecha_retroalimentacion=timezone.now())
for p in pregs_v:
    RetroalimentacionAplicada.objects.create(retroalimentacion=retro_m1, content_type=ct_v, object_id=visita.id, pregunta=p, cantidad_estrellas=4.0)
creadas_ots.append(ot_multi1.id)

ot_multi2 = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT mixta 2", estado="completada")
DetalleTrabajo.objects.create(orden=ot_multi2, nombre="Cmp", descripcion="", content_type=ct_cmp, trabajo_id=compra.id, estado="completado", insumo=next_insumo_disponible())
DetalleTrabajo.objects.create(orden=ot_multi2, nombre="Cot", descripcion="", content_type=ct_cot, trabajo_id=cot.id, estado="completado")
DetalleTrabajo.objects.create(orden=ot_multi2, nombre="Vis", descripcion="", content_type=ct_vis, trabajo_id=visita.id, estado="completado")
retro_m2 = Retroalimentacion.objects.create(orden_trabajo=ot_multi2, usuario_empresa=usuario_emp, observacion_retroalimentacion="OK", fecha_retroalimentacion=timezone.now())
for p in pregs_v:
    RetroalimentacionAplicada.objects.create(retroalimentacion=retro_m2, content_type=ct_v, object_id=visita.id, pregunta=p, cantidad_estrellas=5.0)
creadas_ots.append(ot_multi2.id)

print("OTs creadas:", creadas_ots)

print("\n="*40)
print("✅ VALIDACIÓN Y CIERRE")
print("="*80)
for oid in creadas_ots:
    res = validar_cierre_ot(oid)
    print(f"OT #{oid}: puede_cerrar={res['puede_cerrar']} -> {res['validaciones']}")
    if res['observaciones']:
        print("  Observaciones:")
        for o in res['observaciones']:
            print("   -", o)
    if res['puede_cerrar']:
        cierre = cerrar_ot(oid, comentario="Cierre automático coh." )
        print(f"  Cierre creado: valido={cierre.valido} id={cierre.id}")
print("Hecho.")

🔎 AUDITORÍA DE DATOS INICIALES
Sucursal prestador: 1
Bodega: 1
Proveedor: 2 | Item: 1
Preguntas creadas (si faltaban): 0

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
🔧 CREACIÓN/REUTILIZACIÓN DE ENTIDADES ASOCIADAS
Cotización: 4
Compra: 8 | Ítems: 1
Guías disponibles antes: 1
Visita: 1

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
🧱 CREACIÓN DE OTs DE EJEMPLO
OTs creadas: [21, 22, 23, 24, 25]

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
✅ VALIDACIÓN Y CIERRE
OT #21: puede_cerrar=True -> {'cotizacion': True, 'visitasoporte': True, 'compra': True}
  Cierre creado: valido=True id=6
OT #22: puede_cerrar=True -> {'cotizacion': True, 'visitasoporte': True, 'compra': True}
  Cierre creado: valido=True id=7
OT #23: puede_cerrar=True -> {'cotizacion': True, 'visitasoporte': True, 'compra': True}
  Cierre creado: valido=True id=8
OT #24: puede_cerrar=True -> {'cotizacion': True, 'visitasopor

In [30]:
# Casos adicionales de prueba: OTs que NO deben poder cerrarse y mixtas parcialmente válidas
from importlib import reload
import ordentrabajo.utils as ot_utils_mod
reload(ot_utils_mod)

from django.utils import timezone
from django.contrib.contenttypes.models import ContentType
from ordentrabajo.models import OrdenDeTrabajo, DetalleTrabajo
from cotizaciones.models import Cotizacion
from visitas.models import VisitaSoporte
from bodegas.models import Compra, ItemEnCompra, GuiaSalida, Bodega
from empresas.models import Empresa
from items.models import ItemEmpresa, Categoria
from ordentrabajo.utils import validar_cierre_ot, cerrar_ot, obtener_insumo_disponible
from retroalimentacion.models import Retroalimentacion, RetroalimentacionAplicada

print("="*90)
print("🧪 CREANDO CASOS DE PRUEBA ADICIONALES")
print("="*90)

prestador = Empresa.objects.order_by('id').first()
cliente = Empresa.objects.exclude(id=prestador.id).first()
suc = prestador.sucursales.first()
usuario_emp = suc.usuarios.first()

bodega = Bodega.objects.filter(sucursal=suc).first() or Bodega.objects.create(nombre="Bodega Test", sucursal=suc)

# Asegurar base mínima de ítems y categoría
cat = Categoria.objects.order_by('id').first() or Categoria.objects.create(nombre="General-Test")
item_base = ItemEmpresa.objects.filter(empresa=prestador).first() or ItemEmpresa.objects.create(nombre="ItemTest", empresa=prestador, categoria=cat)

ct_cot = ContentType.objects.get_for_model(Cotizacion)
ct_vis = ContentType.objects.get_for_model(VisitaSoporte)
ct_cmp = ContentType.objects.get_for_model(Compra)
ct_v = ct_vis

creadas = []

# 1) Cotización sin facturar
cot_no_fact = Cotizacion.objects.create(nombre="Cotización No Facturada", empresa=prestador, cliente=cliente, descripcion="Pendiente factura")
ot_nf = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT con cotización no facturada", estado="completada")
DetalleTrabajo.objects.create(orden=ot_nf, nombre="Det Cot NF", descripcion="Cot no facturada", content_type=ct_cot, trabajo_id=cot_no_fact.id, estado="completado")
creadas.append((ot_nf.id, "cot_no_fact"))

# 2) Visita sin retroalimentación
vis_sin = VisitaSoporte.objects.create(empresa=prestador, cliente=cliente, descripcion_servicio="Visita sin retro", estado="completada")
ot_vs = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT visita sin retro", estado="completada")
DetalleTrabajo.objects.create(orden=ot_vs, nombre="Det Vis SR", descripcion="Visita sin retro", content_type=ct_vis, trabajo_id=vis_sin.id, estado="completado")
creadas.append((ot_vs.id, "visita_sin_retro"))

# 3) Compra sin ítems
compra_sin = Compra.objects.create(sucursal=suc, creado_por=usuario_emp, estado="1", observaciones="Compra sin ítems")
ot_cs = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT compra sin items", estado="completada")
DetalleTrabajo.objects.create(orden=ot_cs, nombre="Det Compra SI", descripcion="Compra sin items", content_type=ct_cmp, trabajo_id=compra_sin.id, estado="completado", insumo=obtener_insumo_disponible(bodega_id=bodega.id, usuario_empresa_id=usuario_emp.id))
creadas.append((ot_cs.id, "compra_sin_items"))

# 4) Compra con guía inválida
compra_bad_guia = Compra.objects.create(sucursal=suc, creado_por=usuario_emp, estado="1", observaciones="Compra guía inválida")
ItemEnCompra.objects.create(compra=compra_bad_guia, item=item_base, cantidad=1, precio=500)
insumo_invalido = GuiaSalida.objects.create(bodega=bodega, creado_por=usuario_emp, estado="P")
ot_cbg = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT compra guía inválida", estado="completada")
DetalleTrabajo.objects.create(orden=ot_cbg, nombre="Det Compra GI", descripcion="Compra con guía inválida", content_type=ct_cmp, trabajo_id=compra_bad_guia.id, estado="completado", insumo=insumo_invalido)
creadas.append((ot_cbg.id, "compra_guia_invalida"))

# 5) Mixta parcialmente válida (cot facturada + visita sin retro + compra ok)
cot_fact = Cotizacion.objects.create(nombre="Cot Facturada Mixta", empresa=prestador, cliente=cliente, descripcion="Facturada")
cot_fact.fecha_facturacion = timezone.now(); cot_fact.save(update_fields=["fecha_facturacion"])
vis_no_retro = VisitaSoporte.objects.create(empresa=prestador, cliente=cliente, descripcion_servicio="Visita sin retro mixta", estado="completada")
compra_ok = Compra.objects.create(sucursal=suc, creado_por=usuario_emp, estado="1", observaciones="Compra ok")
ItemEnCompra.objects.create(compra=compra_ok, item=item_base, cantidad=2, precio=800)
insumo_ok = obtener_insumo_disponible(bodega_id=bodega.id, usuario_empresa_id=usuario_emp.id)
ot_mix = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT mixta parcial", estado="completada")
DetalleTrabajo.objects.create(orden=ot_mix, nombre="Cot", descripcion="Cot facturada", content_type=ct_cot, trabajo_id=cot_fact.id, estado="completado")
DetalleTrabajo.objects.create(orden=ot_mix, nombre="Vis", descripcion="Vis sin retro", content_type=ct_vis, trabajo_id=vis_no_retro.id, estado="completado")
DetalleTrabajo.objects.create(orden=ot_mix, nombre="Cmp", descripcion="Compra ok", content_type=ct_cmp, trabajo_id=compra_ok.id, estado="completado", insumo=insumo_ok)
creadas.append((ot_mix.id, "mixta_parcial"))

# 6) Mixta totalmente inválida
cot_nf2 = Cotizacion.objects.create(nombre="Cot NF Mixta", empresa=prestador, cliente=cliente, descripcion="No facturada")
vis_nf2 = VisitaSoporte.objects.create(empresa=prestador, cliente=cliente, descripcion_servicio="Visita sin retro totalmente", estado="completada")
compra_sin2 = Compra.objects.create(sucursal=suc, creado_por=usuario_emp, estado="1", observaciones="Compra vacía 2")
ot_mix_bad = OrdenDeTrabajo.objects.create(empresa=prestador, cliente=cliente, descripcion="OT mixta totalmente inválida", estado="completada")
DetalleTrabajo.objects.create(orden=ot_mix_bad, nombre="Cot", descripcion="Cot NF", content_type=ct_cot, trabajo_id=cot_nf2.id, estado="completado")
DetalleTrabajo.objects.create(orden=ot_mix_bad, nombre="Vis", descripcion="Vis NF", content_type=ct_vis, trabajo_id=vis_nf2.id, estado="completado")
DetalleTrabajo.objects.create(orden=ot_mix_bad, nombre="Cmp", descripcion="Compra sin items", content_type=ct_cmp, trabajo_id=compra_sin2.id, estado="completado", insumo=obtener_insumo_disponible(bodega_id=bodega.id, usuario_empresa_id=usuario_emp.id))
creadas.append((ot_mix_bad.id, "mixta_invalida"))

print("OTs creadas casos adicionales:", creadas)
print("\nValidaciones:")
for oid, etiqueta in creadas:
    r = validar_cierre_ot(oid)
    print(f"[{etiqueta}] OT {oid} -> puede_cerrar={r['puede_cerrar']} | {r['validaciones']}")
    if r['observaciones']:
        for obs in r['observaciones']:
            print("   -", obs)
    if r['puede_cerrar']:
        cierre = cerrar_ot(oid, comentario=f"Cierre auto caso {etiqueta}")
        print(f"   Cierre creado id={cierre.id} valido={cierre.valido}")
print("Fin casos adicionales.")

🧪 CREANDO CASOS DE PRUEBA ADICIONALES
OTs creadas casos adicionales: [(29, 'cot_no_fact'), (30, 'visita_sin_retro'), (31, 'compra_sin_items'), (32, 'compra_guia_invalida'), (33, 'mixta_parcial'), (34, 'mixta_invalida')]

Validaciones:
[cot_no_fact] OT 29 -> puede_cerrar=True | {'cotizacion': True, 'visitasoporte': True, 'compra': True}
   Cierre creado id=11 valido=True
[visita_sin_retro] OT 30 -> puede_cerrar=False | {'cotizacion': True, 'visitasoporte': False, 'compra': True}
   - Sin retroalimentación registrada o sin respuestas.
[compra_sin_items] OT 31 -> puede_cerrar=False | {'cotizacion': True, 'visitasoporte': True, 'compra': False}
   - Compra sin ítems asociados.
[compra_guia_invalida] OT 32 -> puede_cerrar=False | {'cotizacion': True, 'visitasoporte': True, 'compra': False}
   - Guía de salida no entregada/terminada.
[mixta_parcial] OT 33 -> puede_cerrar=False | {'cotizacion': True, 'visitasoporte': False, 'compra': True}
   - Sin retroalimentación registrada o sin respuesta

In [31]:
# Intento de cierre para OTs de prueba creadas en este notebook
from ordentrabajo.models import OrdenDeTrabajo, CierreAdministrativoOT
from ordentrabajo.utils import validar_cierre_ot, cerrar_ot

# Selecciona OTs con descripción que comienza por 'OT' y que aún no tienen cierre
cerradas_ids = set(CierreAdministrativoOT.objects.values_list('orden_id', flat=True))
ots = (
    OrdenDeTrabajo.objects
    .filter(descripcion__startswith="OT")
    .exclude(id__in=cerradas_ids)
    .order_by('id')
)

print(f"Se encontraron {ots.count()} OTs de prueba sin cierre previo.")
resumen = []
for o in ots:
    r = validar_cierre_ot(o.id)
    print(f"OT {o.id} -> puede_cerrar={r['puede_cerrar']} | {r['validaciones']}")
    if r['observaciones']:
        for obs in r['observaciones']:
            print("  -", obs)
    if r['puede_cerrar']:
        cierre = cerrar_ot(o.id, comentario="Cierre automático desde celda final")
        print(f"  Cierre creado id={cierre.id} valido={cierre.valido}")
        resumen.append((o.id, True, cierre.id))
    else:
        resumen.append((o.id, False, None))

print("\nResumen:")
for oid, ok, cid in resumen:
    if ok:
        print(f"OT {oid}: CERRADA (cierre id={cid})")
    else:
        print(f"OT {oid}: NO CERRADA")

Se encontraron 16 OTs de prueba sin cierre previo.
OT 8 -> puede_cerrar=False | {'cotizacion': True, 'visitasoporte': False, 'compra': False}
  - Sin retroalimentación registrada o sin respuestas.
  - Compra no está en estado 'Completada'.; Compra sin ítems asociados.
OT 9 -> puede_cerrar=False | {'cotizacion': True, 'visitasoporte': False, 'compra': False}
  - Compra no está en estado 'Completada'.; Compra sin ítems asociados.
  - Sin retroalimentación registrada o sin respuestas.
OT 10 -> puede_cerrar=False | {'cotizacion': True, 'visitasoporte': True, 'compra': False}
  - Compra no está en estado 'Completada'.; Compra sin ítems asociados.
OT 11 -> puede_cerrar=False | {'cotizacion': True, 'visitasoporte': True, 'compra': False}
  - Compra no está en estado 'Completada'.; Compra sin ítems asociados.
OT 17 -> puede_cerrar=True | {'cotizacion': True, 'visitasoporte': True, 'compra': True}
  Cierre creado id=12 valido=True
OT 18 -> puede_cerrar=True | {'cotizacion': True, 'visitasoporte

In [32]:
# Cierre forzado para OTs no cerrables (crea cierres inválidos para pruebas)
from ordentrabajo.models import OrdenDeTrabajo, CierreAdministrativoOT
from ordentrabajo.utils import validar_cierre_ot, cerrar_ot

ots = OrdenDeTrabajo.objects.filter(descripcion__startswith="OT").order_by("id")
forzadas = []

print("Buscando OTs de prueba no cerrables para cierre forzado...")
for o in ots:
    # Evitar sobreescribir cierres válidos
    existente = getattr(o, "cierre_administrativo", None)
    if existente and existente.valido:
        continue
    r = validar_cierre_ot(o.id)
    if not r["puede_cerrar"]:
        cierre = cerrar_ot(o.id, comentario="Cierre forzado de prueba", forzar=True)
        forzadas.append((o.id, cierre.id, cierre.valido))
        print(f"OT {o.id}: cierre forzado creado id={cierre.id} valido={cierre.valido}")

if not forzadas:
    print("No había OTs pendientes para cierre forzado o ya tenían cierres válidos.")
else:
    print("\nResumen cierres forzados:")
    for oid, cid, valido in forzadas:
        print(f"- OT {oid}: cierre {cid}, valido={valido}")

Buscando OTs de prueba no cerrables para cierre forzado...
OT 8: cierre forzado creado id=17 valido=False
OT 9: cierre forzado creado id=18 valido=False
OT 10: cierre forzado creado id=19 valido=False
OT 11: cierre forzado creado id=20 valido=False
OT 20: cierre forzado creado id=21 valido=False
OT 27: cierre forzado creado id=22 valido=False
OT 30: cierre forzado creado id=23 valido=False
OT 31: cierre forzado creado id=24 valido=False
OT 32: cierre forzado creado id=25 valido=False
OT 33: cierre forzado creado id=26 valido=False
OT 34: cierre forzado creado id=27 valido=False

Resumen cierres forzados:
- OT 8: cierre 17, valido=False
- OT 9: cierre 18, valido=False
- OT 10: cierre 19, valido=False
- OT 11: cierre 20, valido=False
- OT 20: cierre 21, valido=False
- OT 27: cierre 22, valido=False
- OT 30: cierre 23, valido=False
- OT 31: cierre 24, valido=False
- OT 32: cierre 25, valido=False
- OT 33: cierre 26, valido=False
- OT 34: cierre 27, valido=False


In [33]:
# Reporte resumido de OTs de prueba: estado de cierre y observaciones
from ordentrabajo.models import OrdenDeTrabajo, CierreAdministrativoOT
from ordentrabajo.utils import validar_cierre_ot

# Tomamos OTs creadas desde el notebook (descripcion que comienza con "OT")
ots = OrdenDeTrabajo.objects.filter(descripcion__startswith="OT").order_by('id')
rows = []
for o in ots:
    cierre = getattr(o, 'cierre_administrativo', None)
    if cierre:
        res = cierre.resultado or {}
        valid = cierre.valido
        obs = (res.get('observaciones') or [])
        valids = res.get('validaciones') or {}
        cierre_id = cierre.id
        tiene_cierre = True
    else:
        r = validar_cierre_ot(o.id)
        valid = r['puede_cerrar']
        obs = r['observaciones']
        valids = r['validaciones']
        cierre_id = None
        tiene_cierre = False
    rows.append({
        'ot_id': o.id,
        'descripcion': o.descripcion,
        'tiene_cierre': tiene_cierre,
        'cierre_id': cierre_id,
        'cierre_valido': bool(valid),
        'valid_cotizacion': bool(valids.get('cotizacion', False)),
        'valid_visita': bool(valids.get('visitasoporte', False)),
        'valid_compra': bool(valids.get('compra', False)),
        'observaciones': '; '.join(obs) if obs else ''
    })

# Mostrar como DataFrame si pandas está disponible, si no, impresión simple
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    # Ordenar por cierre_valido desc y luego por ot_id
    df = df.sort_values(['cierre_valido','ot_id'], ascending=[False, True]).reset_index(drop=True)
    display(df)
except Exception:
    for r in rows:
        print(r)

print(f"Total OTs: {len(rows)} | Con cierre: {sum(1 for r in rows if r['tiene_cierre'])} | Cierres válidos: {sum(1 for r in rows if r['cierre_valido'])}")

,ot_id,descripcion,tiene_cierre,cierre_id,cierre_valido,valid_cotizacion,valid_visita,valid_compra,observaciones
0,12,OT solo cotización,True,1,True,True,True,True,
1,13,OT solo visita,True,2,True,True,True,True,
2,14,OT solo compra,True,3,True,True,True,True,
3,15,OT mixta 1,True,4,True,True,True,True,
4,16,OT mixta 2,True,5,True,True,True,True,
5,17,OT solo cotización,True,12,True,True,True,True,
6,18,OT solo visita,True,13,True,True,True,True,
7,19,OT solo compra,True,14,True,True,True,True,
8,21,OT solo cotización,True,6,True,True,True,True,
9,22,OT solo visita,True,7,True,True,True,True,


Total OTs: 27 | Con cierre: 27 | Cierres válidos: 16


In [36]:
# Prueba del flujo via APIClient (DRF) sin JWT, autenticando con force_authenticate
from rest_framework.test import APIClient
from cuentas.models import User
from ordentrabajo.models import OrdenDeTrabajo
from core.models import PersonalizacionUsuario
from empresas.models import SucursalEmpresa

client = APIClient()
user = User.objects.filter(is_active=True).first()
client.force_authenticate(user=user)

# Asegurar que el usuario tenga PersonalizacionUsuario con sucursal_principal
suc = SucursalEmpresa.objects.first()
if suc:
    PersonalizacionUsuario.objects.update_or_create(
        usuario=user,
        defaults={
            "sucursal_principal": suc,
            "tema": "3",
            "font_size": 14,
        }
    )

# Seleccionamos algunas OTs de prueba (creadas por este notebook)
ots_prueba = list(OrdenDeTrabajo.objects.filter(descripcion__startswith="OT").order_by('id').values_list('id', flat=True))[-5:]
print("OTs a probar:", ots_prueba)

for oid in ots_prueba:
    url_base = f"/api/ordenes-trabajo/{oid}"
    r1 = client.get(f"{url_base}/validar-cierre/")
    print(f"[validar] OT {oid} -> {r1.status_code}")
    if r1.status_code == 200:
        data = r1.json()
        print("  puede_cerrar:", data.get('puede_cerrar'), "validaciones:", data.get('validaciones'))

    # Intentar cerrar forzando para simplificar pruebas (idempotente/overwrite)
    r2 = client.post(f"{url_base}/cerrar/", {"comentario": "Prueba APIClient", "forzar": True}, format='json')
    print(f"[cerrar]  OT {oid} -> {r2.status_code}")
    if r2.status_code in (200, 201):
        print("  cierre:", {k: r2.json().get(k) for k in ("id", "valido")})

    r3 = client.get(f"{url_base}/cierre/")
    print(f"[cierre]  OT {oid} -> {r3.status_code}")
    if r3.status_code == 200:
        print("  cierre_valido:", r3.json().get('valido'))

print("Fin prueba APIClient.")

OTs a probar: [30, 31, 32, 33, 34]
[validar] OT 30 -> 200
  puede_cerrar: False validaciones: {'cotizacion': True, 'visitasoporte': False, 'compra': True}
[cerrar]  OT 30 -> 200
  cierre: {'id': 23, 'valido': False}
[cierre]  OT 30 -> 200
  cierre_valido: False
[validar] OT 31 -> 200
  puede_cerrar: False validaciones: {'cotizacion': True, 'visitasoporte': True, 'compra': False}
[cerrar]  OT 31 -> 200
  cierre: {'id': 24, 'valido': False}
[cierre]  OT 31 -> 200
  cierre_valido: False
[validar] OT 32 -> 200
  puede_cerrar: False validaciones: {'cotizacion': True, 'visitasoporte': True, 'compra': False}
[cerrar]  OT 32 -> 200
  cierre: {'id': 25, 'valido': False}
[cierre]  OT 32 -> 200
  cierre_valido: False
[validar] OT 33 -> 200
  puede_cerrar: False validaciones: {'cotizacion': True, 'visitasoporte': False, 'compra': True}
[cerrar]  OT 33 -> 200
  cierre: {'id': 26, 'valido': False}
[cierre]  OT 33 -> 200
  cierre_valido: False
[validar] OT 34 -> 200
  puede_cerrar: False validaciones